# MNIST Digit Recognition + Fashion-MNIST Assignment

CSCI 6379 · Topic 17. Part 1 is the worked example: an MLP that recognizes handwritten digits (MNIST). Part 2 is the assignment starter: run an architecture study on Fashion-MNIST.

## Part 1 — Example: MNIST digit recognition

Load MNIST with a DataLoader, define an MLP, train, and evaluate.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
train_set = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_set  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=128,  shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=1000, shuffle=False)

In [ ]:
import matplotlib.pyplot as plt
# visualize a few samples
imgs, labels = next(iter(train_loader))
for i in range(6):
    plt.subplot(1, 6, i+1); plt.imshow(imgs[i].squeeze(), cmap="gray")
    plt.title(labels[i].item()); plt.axis("off")
plt.show()

In [ ]:
class MNISTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
    def forward(self, x):
        x = x.view(-1, 28*28)              # flatten
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)                 # raw scores (logits)

model = MNISTModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def evaluate(m, loader):
    m.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (m(x).argmax(1) == y).sum().item(); total += y.size(0)
    m.train(); return 100*correct/total

for epoch in range(5):
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
    print(f"epoch {epoch+1}: loss {loss.item():.4f}  test acc {evaluate(model, test_loader):.2f}%")

In [ ]:
# error analysis: look at a few misclassified digits
model.eval(); shown = 0
with torch.no_grad():
    for x, y in test_loader:
        p = model(x.to(device)).argmax(1).cpu()
        for i in range(len(y)):
            if p[i] != y[i] and shown < 6:
                plt.subplot(1, 6, shown+1); plt.imshow(x[i].squeeze(), cmap="gray")
                plt.title(f"pred {p[i].item()}\ntrue {y[i].item()}"); plt.axis("off"); shown += 1
        if shown >= 6: break
plt.show()

## Part 2 — Assignment: architecture study on Fashion-MNIST

Fashion-MNIST has the SAME format as MNIST (28x28, 10 classes) — just swap the dataset. Your task is to investigate **how the network architecture affects test accuracy**. Vary at least two of: number of hidden layers (depth), neurons per layer (width), activation function. Hold everything else fixed. Report a table, a plot, and a written analysis.

Below is a starter: a model builder you can configure, plus a loop skeleton to fill in.

In [ ]:
# Fashion-MNIST: identical pipeline, different dataset
f_train = datasets.FashionMNIST(root="./data", train=True,  download=True, transform=transform)
f_test  = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)
f_train_loader = DataLoader(f_train, batch_size=128,  shuffle=True)
f_test_loader  = DataLoader(f_test,  batch_size=1000, shuffle=False)
CLASSES = ["T-shirt","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Ankle boot"]

In [ ]:
def build_mlp(hidden_sizes, activation=nn.ReLU):
    # hidden_sizes: e.g. [512, 256] for two hidden layers
    layers, in_dim = [], 28*28
    for h in hidden_sizes:
        layers += [nn.Linear(in_dim, h), activation()]
        in_dim = h
    layers += [nn.Linear(in_dim, 10)]
    return nn.Sequential(nn.Flatten(), *layers)

def train_and_score(model, epochs=5, lr=1e-3):
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr); crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for x, y in f_train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); crit(model(x), y).backward(); opt.step()
    return evaluate(model, f_test_loader)

In [ ]:
# EXAMPLE experiment: vary DEPTH (fill in / extend for your assignment study)
results = {}
for hidden in [[256], [256, 256], [256, 256, 256]]:
    acc = train_and_score(build_mlp(hidden))
    results[str(hidden)] = acc
    print(f"hidden layers {hidden}: test acc {acc:.2f}%")

# TODO for the assignment:
#   - also vary WIDTH (e.g. [64], [128], [256], [512]) and/or ACTIVATION (nn.ReLU, nn.Sigmoid, nn.Tanh)
#   - collect results into a table, PLOT accuracy vs architecture with matplotlib
#   - write an analysis: does depth/width help? diminishing returns? overfitting? best activation?